In [29]:
from pathlib import Path
import sys
import importlib
from collections import Counter
import pandas as pd

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "parser.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root containing src/parser.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.normalize
import src.parser
importlib.reload(src.normalize)
importlib.reload(src.parser)
from src.parser import split_ingredients, parse_ingredient

df = pd.read_csv(ROOT / "data" / "processed" / "food_safety_clean (1).csv")
df["items"] = df["ingredients_clean"].fillna("").map(split_ingredients)
df["canon3"] = df["items"].map(
    lambda items: [canonical for item in items for canonical in parse_ingredient(item)]
)
mentions = [name for ingredients in df["canon3"] for name in ingredients]

In [30]:
nut = ["sugars_g", "salt_g", "saturated_fat_g", "energy_kcal"]
ex = df.explode("canon3")
hint = (ex.groupby("canon3")[nut].median().round(1)
          .join(ex["canon3"].value_counts().rename("n")))
hint = hint[hint["n"] >= 15].sort_values("n", ascending=False).head(100)
hint.to_csv("../data/processed/ingredient_nutrient_hints.csv")
hint.head(30)

,sugars_g,salt_g,saturated_fat_g,energy_kcal,n
canon3,,,,,
salt,4.7,1.1,5.1,388.6,753
sugar,16.4,0.6,4.4,379.4,643
added flavour,15.5,0.5,5.5,417.0,559
emulsifier,24.3,0.5,9.0,445.0,378
acidity regulator,7.4,1.2,2.1,350.0,350
milk solids,21.0,0.4,7.7,395.0,297
water,4.2,0.6,0.7,119.0,239
ins 330,6.3,1.0,4.1,379.4,227
colour,13.7,0.5,2.5,329.6,223


In [31]:
kb = pd.read_csv("../data/knowledgebase/ingredient.csv")
known = set(kb["ingredient"])
mentions = [x for L in df["canon3"] for x in L]
hit = sum(1 for m in mentions if m in known)
print(f"KB recognizes {hit/len(mentions):.1%} of ingredient mentions")
for name, n in Counter(m for m in mentions if m not in known).most_common(30):
    print(n, name)

KB recognizes 52.5% of ingredient mentions
96 red chilli
42 cinnamon
36 clove
34 cardamom
34 nutmeg
32 ins 476
31 tomato paste
31 potato
31 rice
30 almonds
30 ins 331
29 asafoetida
29 carbonated water
27 ins 296
27 ins 339
27 chilli
27 edible vegetable oil palmolein
27 dates
27 spices
25 edible vegetable oil palm
25 ins 319
25 ins 440
24 ins 501
24 soya flour
24 green chilli
24 ins 422
24 ins 200
23 corn
23 tomato
23 curry leaves


In [32]:
a = pd.read_csv("../data/knowledgebase/ingredient.csv")
b = pd.read_csv("../data/knowledgebase/ingredients_batch3.csv")
kb = pd.concat([a, b]).drop_duplicates("ingredient", keep="last")
kb.to_csv("../data/knowledgebase/ingredient.csv", index=False)
print(len(kb), "rows")

98 rows


In [33]:
from src.kb import load_kb, lookup
kb = load_kb()
how = Counter(lookup(m, kb)[1].split(":")[0] for m in mentions)
total = sum(how.values())
print({k: f"{v/total:.1%}" for k, v in how.items()})

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\HP\\Desktop\\food-safety-advisor\\data\\knowledge_base\\ingredients.csv'

In [ ]:
unk = Counter(m for m in mentions if lookup(m, kb)[1] == "unrecognized")
tot = len(mentions)
recognized = tot - sum(unk.values())

cum = 0
marks = {50, 100, 200, 400, 800}
for i, (name, n) in enumerate(unk.most_common(), 1):
    cum += n
    if i in marks:
        print(f"+{i} new rows -> coverage {(recognized + cum)/tot:.1%}")

print(len(unk), "distinct unrecognized names in total")
for name, n in unk.most_common(60):
    print(n, name)

+50 new rows -> coverage 57.0%
+100 new rows -> coverage 62.2%
+200 new rows -> coverage 68.5%
+400 new rows -> coverage 75.2%
+800 new rows -> coverage 81.6%
3640 distinct unrecognized names in total
43 garlic powder
40 edible vegetable oil palm oil
40 turmeric powder
38 cinnamon
37 black salt
36 clove
35 onion powder
34 red chilli
34 cardamom
34 nutmeg
33 red chilli powder
33 refined palm oil oil
32 ins 476
31 tomato paste
31 potato
31 rice
30 corn starch
30 almonds
30 ins 331
29 asafoetida
29 edible common salt
29 carbonated water
28 rolled oats
27 ins 296
27 ins 339
27 chilli
27 dates
27 spices
26 cumin powder
25 edible vegetable oil palm
25 wheat flour atta
25 ins 319
25 ins 440
24 ins 501
24 soya flour
24 green chilli
24 ins 422
24 ins 200
24 gluten
23 corn
23 tomato
23 chilli powder
23 curry leaves
23 fenugreek
23 mustard
23 ginger powder
22 rice bran oil
22 ins 122
22 spices condiments
22 cashew nuts
22 vinegar
21 malt extract
21 soy protein isolate
21 wheat bran
21 ins 450
21 

In [ ]:
import importlib, src.kb; importlib.reload(src.kb)
from src.kb import load_kb, lookup
kb = load_kb("../data/knowledgebase/ingredient.csv")

how = Counter(lookup(m, kb)[1].split(":")[0] for m in mentions)
tot = sum(how.values())
print({k: f"{v/tot:.1%}" for k, v in how.items()})

# read what the rules matched. Wrong merges matter more than coverage
import random; random.seed(0)
for kind in ("stripped", "fallback", "whole"):
    sample = [m for m in set(mentions) if lookup(m, kb)[1].startswith(kind)]
    print("\n", kind, len(sample))
    for m in random.sample(sample, min(15, len(sample))):
        print("  ", m, "->", lookup(m, kb)[1])

{'unrecognized': '39.4%', 'exact': '46.1%', 'fallback': '2.3%', 'whole': '10.4%', 'stripped': '1.8%'}

 stripped 35
   dried yeast -> stripped
   dried garlic -> stripped
   turmeric powder -> stripped
   onion powder -> stripped
   garlic flakes -> stripped
   ginger paste -> stripped
   yeast extract powder -> stripped
   cumin powder -> stripped
   ground black pepper -> stripped
   dried glucose syrup -> stripped
   coriander powder -> stripped
   dextrose powder -> stripped
   roasted garlic -> stripped
   ground cumin -> stripped
   dried onion -> stripped

 fallback 202
   ng erients med wheat flour maida -> fallback:any refined starch
   sodium hexametaphosphate -> fallback:any sodium additive
   instant noodles with seasoning redients noodles refined wheat flour maida -> fallback:any refined starch
   hydrogenated vegetable fat coconut -> fallback:any hydrogenated fat
   milk solids sucrose -> fallback:any added sugar
   sodium alginate -> fallback:any sodium additive
   evert

In [ ]:
a = pd.read_csv("../data/knowledgebase/ingredient.csv")
b = pd.read_csv("../data/knowledgebase/ingredients_batch4.csv")
kb = pd.concat([a, b]).drop_duplicates("ingredient", keep="last")
kb.to_csv("../data/knowledgebase/ingredient.csv", index=False)
print(len(kb), "rows")

98 rows


In [ ]:
from pathlib import Path
import sys
import importlib

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "kb.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root containing src/kb.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import src.kb
import src.normalize
import src.parser
from collections import Counter

importlib.reload(src.kb)
importlib.reload(src.normalize)
importlib.reload(src.parser)
from src.kb import load_kb, lookup
from src.parser import split_ingredients, parse_ingredient

df = pd.read_csv(ROOT / "data" / "processed" / "food_safety_clean (1).csv")
df["items"] = df["ingredients_clean"].fillna("").map(split_ingredients)
df["canon3"] = df["items"].map(
    lambda items: [canonical for item in items for canonical in parse_ingredient(item)]
)
mentions = [name for ingredients in df["canon3"] for name in ingredients]
kb = load_kb(ROOT / "data" / "knowledgebase" / "ingredient.csv")
# 1. Overall split
how = Counter(lookup(m, kb)[1].split(":")[0] for m in mentions)
tot = sum(how.values())
print({k: f"{v/tot:.1%}" for k, v in how.items()})

# 2. Risk-relevant coverage (the number that matters for safety)
RISKY = ("sugar", "syrup", "dextrose", "fructose", "glucose", "honey", "jaggery", "molasses",
         "salt", "sodium", "oil", "fat", "butter", "ghee", "hydrogenated",
         "sweetener", "colour", "flavour", "preservative", "ins ")
risky = [m for m in mentions if any(k in m for k in RISKY)]
unk_r = [m for m in risky if lookup(m, kb)[1] == "unrecognized"]
print(f"Risk-relevant: {len(risky)} mentions, unrecognized {len(unk_r)} ({len(unk_r)/len(risky):.1%})")
for name, n in Counter(unk_r).most_common(25):
    print(n, name)

# 3. Read the riskier matches by eye
import random; random.seed(0)
for kind in ("fallback", "whole"):
    sample = [m for m in set(mentions) if lookup(m, kb)[1].startswith(kind)]
    print("\n", kind, len(sample))
    for m in random.sample(sample, min(15, len(sample))):
        print("  ", m, "->", lookup(m, kb)[1])

{'unrecognized': '35.9%', 'exact': '52.5%', 'fallback': '1.7%', 'whole': '9.5%', 'stripped': '0.4%'}
Risk-relevant: 6950 mentions, unrecognized 1941 (27.9%)
32 ins 476
30 ins 331
27 ins 296
27 ins 339
27 edible vegetable oil palmolein
25 edible vegetable oil palm
25 ins 319
25 ins 440
24 ins 501
24 ins 422
24 ins 200
22 ins 122
21 ins 450
21 ins 341
21 ins 481
20 ins 635
19 ins 170
18 ins 451
16 ins 508
16 ins 452
16 ins 960
15 ins 223
15 ins 150c
14 ins 320
13 ins 133

 fallback 187
   refined wheat flour maida 71 -> fallback:any refined starch
   110 outer layer mango flavored coating coconut oil -> fallback:any flavour
   refined wheat flour maida refined palmolein oil -> fallback:any refined starch
   hydrogenated palm fat -> fallback:any hydrogenated fat
   edible vegetable fat hydrogenated -> fallback:any hydrogenated fat
   sodium benzte -> fallback:any sodium additive
   disodium 5 inosinate -> fallback:any sodium additive
   instant noodles refined wheat flour maida -> fallbac

In [ ]:
key = ["sugar", "glucose syrup", "maltodextrin", "refined wheat flour", "salt", "palm oil",
       "hydrogenated vegetable fat", "ins 621", "ins 211", "ins 102", "sweetener", "honey"]
kb.loc[key, ["category", "risk_general", "risk_diabetes", "risk_hypertension",
             "risk_obesity", "reason", "source"]]

,category,risk_general,risk_diabetes,risk_hypertension,risk_obesity,reason,source
ingredient,,,,,,,
sugar,added sugar,2,3,0,2,NaN,WHO free sugars guideline; ADA Standards of Care
glucose syrup,added sugar,2,3,0,2,NaN,WHO free sugars guideline; ADA Standards of Care
maltodextrin,refined carbohydrate,2,3,0,1,NaN,ADA Standards of Care
refined wheat flour,refined grain,1,2,0,1,NaN,ICMR-NIN; ADA Standards of Care
salt,sodium source,2,0,3,0,NaN,WHO sodium guideline; AHA
palm oil,saturated fat,2,0,0,2,NaN,WHO saturated and trans fat guidelines
hydrogenated vegetable fat,trans fat,3,1,1,2,NaN,WHO saturated and trans fat guidelines
ins 621,flavour enhancer,1,0,2,0,NaN,JECFA; FSSAI additive regulations
ins 211,preservative,1,0,1,0,NaN,FSSAI additive regulations


In [ ]:
print(kb.columns.tolist())
print(kb["reason"].isna().sum(), "of", len(kb), "rows have an empty reason")

['category', 'risk_general', 'risk_diabetes', 'risk_hypertension', 'risk_obesity', 'source', 'verified', 'risk_smoker', 'reason']
53 of 98 rows have an empty reason


In [ ]:
from pathlib import Path
import pandas as pd

d = Path("../data/knowledgebase")
parts = [pd.read_csv(d / "ingredient.csv")]
parts += [pd.read_csv(d / f"ingredients_batch{i}.csv") for i in (2, 3, 4)]
kb = pd.concat(parts).drop_duplicates("ingredient", keep="last")
kb.to_csv(f"{d}/ingredients.csv", index=False)
print(len(kb), "rows;", kb["reason"].isna().sum(), "empty reasons")


98 rows; 53 empty reasons


In [ ]:
d = "../data/knowledgebase"
kb = pd.read_csv(f"{d}/ingredients (1).csv").set_index("ingredient")
orig = pd.read_csv(f"{d}/ingredients (1).csv").set_index("ingredient")

fixed = 0
for col in ["reason", "source", "category"]:
    missing = kb[col].isna() & kb.index.isin(orig.index)
    kb.loc[missing, col] = orig.loc[kb.index[missing], col]
    fixed += missing.sum()

kb.reset_index().to_csv(f"{d}/ingredients.csv", index=False)
print(fixed, "cells repaired;", kb["reason"].isna().sum(), "empty reasons left")

0 cells repaired; 0 empty reasons left


In [ ]:
d = "../data/knowledgebase"
kb = pd.read_csv(f"{d}/ingredients.csv")
kb = kb.drop(columns=["risk_smoker"], errors="ignore")
kb.to_csv(f"{d}/ingredients.csv", index=False)
print(kb.columns.tolist())

['ingredient', 'category', 'risk_general', 'risk_diabetes', 'risk_hypertension', 'risk_obesity', 'reason', 'source', 'verified']


In [ ]:
from src.profile import Profile, personal_risk
from src.kb import load_kb
kb = load_kb(ROOT / "data" / "knowledgebase/ingredients.csv")
me = Profile(diabetes=True, height_cm=170, weight_kg=82)
for name in ["sugar", "salt", "palm oil", "oats"]:
    print(name, "general:", kb.loc[name, "risk_general"], "| for this person:", personal_risk(kb.loc[name], me))

sugar general: 2 | for this person: 3
salt general: 2 | for this person: 2
palm oil general: 2 | for this person: 2
oats general: 0 | for this person: 0


In [ ]:
from src.profile import Profile, active_conditions
print(active_conditions(me))            # expect ['diabetes', 'obesity']

for name in ["honey", "maltodextrin"]:
    print(name, "general:", kb.loc[name, "risk_general"], "| for this person:", personal_risk(kb.loc[name], me))

ModuleNotFoundError: No module named 'src'

In [3]:
from src.verdict import analyse
from src.profile import Profile
import json

text = "Refined wheat flour (maida), sugar, palm oil, salt, raising agents (INS 500(ii), INS 503(ii)), emulsifier (INS 322), added flavour"
print(json.dumps(analyse(text, kb, Profile(diabetes=True)), indent=2))

ModuleNotFoundError: No module named 'src'